# ⚽ Previsão de Rebaixamento no Brasileirão Série A
## Trabalho de Conclusão de Curso — Ciência de Dados / UFPB

**Aluno:** Leonardo Feitosa  
**Tema:** Previsão de Rebaixamento no Brasileirão Série A 2025 com Machine Learning  
**Modelo:** Regressão Logística (sklearn)  
**Dados:** Transfermarkt — Temporadas 2014–2025 (240 registros)

---

### Sumário
1. [Importações e Configuração](#1)
2. [Carregamento e Inspeção dos Dados](#2)
3. [Variável Dependente e Variáveis Independentes](#3)
4. [Análise Exploratória dos Dados (EDA)](#4)
5. [Separação Formal Treino / Teste](#5)
6. [Treinamento dos Modelos](#6)
7. [Avaliação no Conjunto de Teste](#7)
8. [Seleção e Salvamento do Modelo Final](#8)
9. [Previsão para 2025](#9)
10. [Análise de Mudança de Situação entre Temporadas](#10)

---
## 1. Importações e Configuração <a id='1'></a>

In [ ]:
# Bibliotecas de manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, mean_absolute_error, mean_squared_error,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Persistência de modelos
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ Bibliotecas importadas com sucesso!')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
import sklearn; print(f'   sklearn {sklearn.__version__}')

---
## 2. Carregamento e Inspeção dos Dados <a id='2'></a>

Os dados foram coletados no site **Transfermarkt** e abrangem as temporadas de **2014 a 2025** do Brasileirão Série A.  
Cada linha representa um clube em uma temporada específica.

In [ ]:
# ── Carregando o arquivo Excel ──────────────────────────────────────
ARQUIVO = os.path.join('dados', 'BASE_FINAL.xlsx')

try:
    df = pd.read_excel(ARQUIVO, sheet_name='CLUBES')
except Exception:
    df = pd.read_excel(ARQUIVO)       # fallback: primeira aba

df.columns = df.columns.str.strip()

print(f'Registros carregados : {len(df)}')
print(f'Colunas              : {list(df.columns)}')
print(f'Temporadas           : {sorted(df["Temporada"].unique())}')
df.head(10)

In [ ]:
# ── Informações gerais do DataFrame ────────────────────────────────
df.info()

In [ ]:
# ── Contagem de registros por temporada ────────────────────────────
contagem = df.groupby('Temporada').size().reset_index(name='Clubes')

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(contagem['Temporada'], contagem['Clubes'], color='steelblue', edgecolor='white')
ax.set_title('Quantidade de Clubes por Temporada', fontsize=14, fontweight='bold')
ax.set_xlabel('Temporada')
ax.set_ylabel('Nº de Clubes')
ax.set_xticks(contagem['Temporada'])
for bar in ax.patches:
    ax.annotate(f'{int(bar.get_height())}',
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print(contagem.to_string(index=False))

In [ ]:
# ── Verificação de valores ausentes ────────────────────────────────
nulos = df.isnull().sum()
print('Valores ausentes por coluna:')
print(nulos[nulos > 0] if nulos.sum() > 0 else '   Nenhum valor ausente encontrado ✅')

---
## 3. Variável Dependente e Variáveis Independentes <a id='3'></a>

### 3.1 Variável Dependente — `Status_bin`

A variável dependente é **binária**:

| Valor | Significado | Critério na coluna `Situacao` |
|:-----:|:------------|:-----------------------------|
| **0** | Rebaixado   | `Situacao == 'Rebaixado'`    |
| **1** | Permaneceu  | Qualquer outro valor         |

### 3.2 Variáveis Independentes (Features)

| Feature                  | Descrição                                              | Unidade |
|:-------------------------|:-------------------------------------------------------|:--------|
| **Plantel**              | Número total de atletas registrados no clube           | Atletas |
| **Estrangeiros**         | Número de atletas estrangeiros no elenco               | Atletas |
| **Valor de Mercado Total** | Soma do valor de mercado de todos os atletas do clube | Milhões € |

In [ ]:
# ── Criando Status_bin ──────────────────────────────────────────────
if 'Situacao' in df.columns:
    df['Status_bin'] = df['Situacao'].apply(
        lambda x: 0 if str(x).strip().lower() == 'rebaixado' else 1
    )
elif 'Status' in df.columns:
    df['Status_bin'] = df['Status'].apply(
        lambda x: 0 if pd.notna(x) and int(x) == 3 else 1
    )

# Distribuição da variável dependente
dist = df['Status_bin'].value_counts().rename({0: 'Rebaixado (0)', 1: 'Permaneceu (1)'})
print('Distribuição de Status_bin:')
print(dist)
print(f'\nProporção de rebaixamentos: {dist["Rebaixado (0)"] / len(df):.1%}')

fig, ax = plt.subplots(figsize=(5, 4))
ax.pie(dist.values, labels=dist.index, autopct='%1.1f%%',
       colors=['#e74c3c', '#2ecc71'], startangle=90,
       wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title('Distribuição da Variável Dependente', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Definição das constantes do modelo ─────────────────────────────
FEATURES         = ['Plantel', 'Estrangeiros', 'Valor de Mercado Total']
TARGET           = 'Status_bin'
ANO_CORTE_TREINO = 2022
ANO_PREVISAO     = 2025

print('Features (variáveis independentes):', FEATURES)
print('Target  (variável dependente)     :', TARGET)
print(f'Período de treino : 2014 – {ANO_CORTE_TREINO}')
print(f'Período de teste  : {ANO_CORTE_TREINO+1} – {ANO_PREVISAO-1}')
print(f'Previsão para     : {ANO_PREVISAO}')

# Estatísticas descritivas das features
df[FEATURES + [TARGET]].describe().round(2)

---
## 4. Análise Exploratória dos Dados (EDA) <a id='4'></a>

In [ ]:
# ── Histogramas por situação ────────────────────────────────────────
labels = {0: 'Rebaixado', 1: 'Permaneceu'}
cores  = {0: '#e74c3c', 1: '#2ecc71'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Distribuição das Features por Situação do Clube', fontsize=14, fontweight='bold')

for ax, feat in zip(axes, FEATURES):
    for status_val, label in labels.items():
        dados = df[df['Status_bin'] == status_val][feat].dropna()
        ax.hist(dados, bins=15, alpha=0.6, label=label,
                color=cores[status_val], edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frequência')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Boxplots: Features vs Situação ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Boxplots: Features por Situação do Clube', fontsize=14, fontweight='bold')

for ax, feat in zip(axes, FEATURES):
    dados_plot = df[['Status_bin', feat]].copy()
    dados_plot['Situação'] = dados_plot['Status_bin'].map(labels)
    sns.boxplot(
        data=dados_plot, x='Situação', y=feat, ax=ax,
        palette={'Rebaixado': '#e74c3c', 'Permaneceu': '#2ecc71'},
        width=0.5
    )
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# ── Médias das features por situação ───────────────────────────────
medias = df.groupby('Status_bin')[FEATURES].mean().round(2)
medias.index = ['Rebaixado', 'Permaneceu']
print('Médias das features por situação:')
medias

In [ ]:
# ── Matriz de correlação ────────────────────────────────────────────
cols_corr = FEATURES + [TARGET]
corr = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=-1, vmax=1, ax=ax, linewidths=0.5,
    annot_kws={'size': 12}
)
ax.set_title('Matriz de Correlação — Features + Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlação de cada feature com Status_bin:')
print(corr['Status_bin'].drop('Status_bin').sort_values(ascending=False))

In [ ]:
# ── Scatterplot: Valor de Mercado vs Pontos, colorido por situação ─
if 'Pontos' in df.columns:
    df_plot = df[df['Temporada'] < ANO_PREVISAO].copy()
    df_plot['Situação'] = df_plot['Status_bin'].map(labels)

    fig, ax = plt.subplots(figsize=(10, 6))
    for sit, cor in [('Rebaixado', '#e74c3c'), ('Permaneceu', '#2ecc71')]:
        sub = df_plot[df_plot['Situação'] == sit]
        ax.scatter(sub['Valor de Mercado Total'], sub['Pontos'],
                   c=cor, alpha=0.6, label=sit, edgecolors='white', s=60)

    ax.set_xlabel('Valor de Mercado Total (M€)', fontsize=12)
    ax.set_ylabel('Pontos', fontsize=12)
    ax.set_title('Valor de Mercado vs Pontos — por Situação', fontsize=13, fontweight='bold')
    ax.legend()
    ax.axhline(y=45, color='gray', linestyle='--', alpha=0.5, label='Limiar ~45 pts')
    plt.tight_layout()
    plt.show()

---
## 5. Separação Formal Treino / Teste <a id='5'></a>

### Por que NÃO usar separação aleatória em séries temporais?

Em problemas de **séries temporais**, dividir os dados aleatoriamente introduz **vazamento de informação do futuro** (*data leakage*):  
o modelo "vê" eventos futuros durante o treino, o que infla artificialmente as métricas e torna o modelo inútil na prática.

**Separação adotada neste trabalho:**

| Conjunto   | Período        | Uso                        |
|:-----------|:--------------|:---------------------------|
| **Treino** | 2014 – 2022   | Ajuste dos modelos          |
| **Teste**  | 2023 – 2024   | Avaliação fora da amostra   |
| **Previsão** | 2025        | Predição real sem rótulo    |

In [ ]:
# ── Separação por corte temporal ────────────────────────────────────
df_rotulado = df[df['Temporada'] < ANO_PREVISAO].copy()
df_treino   = df_rotulado[df_rotulado['Temporada'] <= ANO_CORTE_TREINO].copy()
df_teste    = df_rotulado[df_rotulado['Temporada']  > ANO_CORTE_TREINO].copy()
df_prev     = df[df['Temporada'] == ANO_PREVISAO].copy()

print('─' * 50)
print(f'  Treino  (2014–{ANO_CORTE_TREINO}): {len(df_treino):>4} registros')
print(f'  Teste   ({ANO_CORTE_TREINO+1}–{ANO_PREVISAO-1}) : {len(df_teste):>4} registros')
print(f'  Previsão ({ANO_PREVISAO})  : {len(df_prev):>4} registros')
print('─' * 50)

# Separação de X e y
X_treino = df_treino[FEATURES]
y_treino = df_treino[TARGET]
X_teste  = df_teste[FEATURES]
y_teste  = df_teste[TARGET]

print(f'\nDistribuição no TREINO:')
print(y_treino.value_counts().rename({0: 'Rebaixado', 1: 'Permaneceu'}))
print(f'\nDistribuição no TESTE:')
print(y_teste.value_counts().rename({0: 'Rebaixado', 1: 'Permaneceu'}))

In [ ]:
# ── Visualização da divisão temporal ───────────────────────────────
temporadas_treino = sorted(df_treino['Temporada'].unique())
temporadas_teste  = sorted(df_teste['Temporada'].unique())

fig, ax = plt.subplots(figsize=(12, 2.5))
for t in temporadas_treino:
    ax.barh(0, 1, left=t - 2014, color='#2980b9', height=0.5)
for t in temporadas_teste:
    ax.barh(0, 1, left=t - 2014, color='#e67e22', height=0.5)

ax.set_xlim(-0.5, 12)
ax.set_xticks(range(12))
ax.set_xticklabels(range(2014, 2026))
ax.set_yticks([])
ax.set_title('Divisão Temporal: Treino vs Teste', fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#2980b9', label=f'Treino (2014–{ANO_CORTE_TREINO})'),
    Patch(color='#e67e22', label=f'Teste ({ANO_CORTE_TREINO+1}–{ANO_PREVISAO-1})')
], loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# ── Normalização: StandardScaler ajustado APENAS no treino ─────────
scaler = StandardScaler()
X_treino_sc = scaler.fit_transform(X_treino)   # fit + transform no treino
X_teste_sc  = scaler.transform(X_teste)        # apenas transform no teste

print('Médias do Scaler (calculadas no treino):')
for feat, mean, std in zip(FEATURES, scaler.mean_, scaler.scale_):
    print(f'  {feat:<30}  média={mean:.2f}  desvio={std:.2f}')

print('\nPrimeiras 3 linhas — treino normalizado:')
print(pd.DataFrame(X_treino_sc, columns=FEATURES).head(3).round(3))

---
## 6. Treinamento dos Modelos <a id='6'></a>

São treinados **3 modelos** com os mesmos dados de treino para comparação:

| Modelo                | Características                                            |
|:----------------------|:-----------------------------------------------------------|
| **Regressão Logística** | Modelo linear, interpretável, coeficientes disponíveis  |
| **Random Forest**     | Ensemble de árvores, robusto a outliers                   |
| **SVM**               | Margem máxima, bom para espaços de alta dimensão          |

In [ ]:
# ── Instanciação dos modelos ────────────────────────────────────────
modelos = {
    'Regressão Logística': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':                 SVC(probability=True, random_state=42)
}

modelos_treinados = {}

for nome, modelo in modelos.items():
    modelo.fit(X_treino_sc, y_treino)
    modelos_treinados[nome] = modelo
    print(f'✅ {nome} — treinado com {len(X_treino_sc)} exemplos')

In [ ]:
# ── Verificar coeficientes da Regressão Logística (treino inicial) ──
lr = modelos_treinados['Regressão Logística']

coef_df = pd.DataFrame({
    'Feature':     FEATURES,
    'Coeficiente': lr.coef_[0].round(4)
}).sort_values('Coeficiente', ascending=False)

print('Coeficientes da Regressão Logística (conjunto de treino):')
print(coef_df.to_string(index=False))
print(f'\nIntercepto: {lr.intercept_[0]:.4f}')
print('\nInterpretação:')
print('  > Coef. positivo → aumenta prob. de permanência (classe 1)')
print('  > Coef. negativo → aumenta prob. de rebaixamento (classe 0)')

# Gráfico de coeficientes
fig, ax = plt.subplots(figsize=(8, 3))
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coeficiente']]
ax.barh(coef_df['Feature'], coef_df['Coeficiente'], color=colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Coeficientes — Regressão Logística', fontweight='bold')
ax.set_xlabel('Coeficiente')
for bar, val in zip(ax.patches, coef_df['Coeficiente']):
    ax.text(val + 0.01 * np.sign(val), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

---
## 7. Avaliação no Conjunto de Teste <a id='7'></a>

In [ ]:
# ── Métricas dos 3 modelos no conjunto de teste ─────────────────────
resultados = []

for nome, modelo in modelos_treinados.items():
    y_pred = modelo.predict(X_teste_sc)
    resultados.append({
        'Modelo':    nome,
        'Acurácia':  accuracy_score(y_teste, y_pred),
        'MAE':       mean_absolute_error(y_teste, y_pred),
        'RMSE':      np.sqrt(mean_squared_error(y_teste, y_pred))
    })

df_metricas = pd.DataFrame(resultados).set_index('Modelo')
print('Métricas no Conjunto de Teste (2023–2024):')
df_metricas.style.format({'Acurácia': '{:.4f}', 'MAE': '{:.4f}', 'RMSE': '{:.4f}'})\
                 .background_gradient(cmap='Greens', subset=['Acurácia'])\
                 .background_gradient(cmap='Reds_r', subset=['MAE', 'RMSE'])

In [ ]:
# ── Gráfico comparativo das métricas ───────────────────────────────
df_metricas_plot = df_metricas.reset_index().melt(id_vars='Modelo', var_name='Métrica', value_name='Valor')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Comparação de Métricas — Conjunto de Teste', fontsize=14, fontweight='bold')

cores_modelos = ['#2980b9', '#27ae60', '#8e44ad']

for ax, metrica in zip(axes, ['Acurácia', 'MAE', 'RMSE']):
    sub = df_metricas_plot[df_metricas_plot['Métrica'] == metrica]
    bars = ax.bar(sub['Modelo'], sub['Valor'], color=cores_modelos, edgecolor='white', width=0.5)
    ax.set_title(metrica, fontweight='bold')
    ax.set_xticklabels(sub['Modelo'], rotation=15, ha='right', fontsize=9)
    ax.set_ylim(0, 1.1 if metrica == 'Acurácia' else None)
    for bar in bars:
        ax.annotate(f'{bar.get_height():.3f}',
                    (bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Classification Report — Regressão Logística ─────────────────────
print('Classification Report — Regressão Logística')
print('=' * 55)
y_pred_lr = modelos_treinados['Regressão Logística'].predict(X_teste_sc)
print(classification_report(
    y_teste, y_pred_lr,
    target_names=['Rebaixado (0)', 'Permaneceu (1)']
))

In [ ]:
# ── Matrizes de Confusão dos 3 modelos ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Matrizes de Confusão — Conjunto de Teste (2023–2024)',
             fontsize=14, fontweight='bold')

for ax, (nome, modelo) in zip(axes, modelos_treinados.items()):
    y_pred = modelo.predict(X_teste_sc)
    cm = confusion_matrix(y_teste, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Rebaixado', 'Permaneceu'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nome, fontweight='bold', fontsize=11)
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Real')

plt.tight_layout()
os.makedirs('img', exist_ok=True)
plt.savefig(os.path.join('img', 'matrizes_confusao.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Salvo em img/matrizes_confusao.png')

In [ ]:
# ── Probabilidades previstas vs real (Regressão Logística) ──────────
probs_teste = modelos_treinados['Regressão Logística'].predict_proba(X_teste_sc)[:, 0]
df_probs = df_teste[['Clube', 'Temporada', 'Pontos']].copy()
df_probs['Prob_Rebaixamento'] = probs_teste
df_probs['Real']              = y_teste.map({0: 'Rebaixado', 1: 'Permaneceu'}).values
df_probs['Previsto']          = (y_pred_lr == 0).astype(int).map({1: 'Rebaixado', 0: 'Permaneceu'})

print('Probabilidades de Rebaixamento no Conjunto de Teste:')
df_probs.sort_values('Prob_Rebaixamento', ascending=False).head(15)

---
## 8. Seleção e Salvamento do Modelo Final <a id='8'></a>

O modelo selecionado é a **Regressão Logística**, pois:
- Apresenta boa acurácia no conjunto de teste
- É **interpretável**: os coeficientes revelam a influência de cada feature
- É computacionalmente eficiente

O modelo final é retreinado com **todos os dados rotulados (2014–2024)** para maximizar a informação disponível antes da previsão de 2025.

In [ ]:
# ── Retreinamento com todos os dados rotulados (2014–2024) ──────────
X_todos = df_rotulado[FEATURES]
y_todos = df_rotulado[TARGET]

# Novo scaler ajustado em todos os dados rotulados
scaler_final = StandardScaler()
X_todos_sc   = scaler_final.fit_transform(X_todos)

# Modelo final
modelo_final = LogisticRegression(random_state=42, max_iter=1000)
modelo_final.fit(X_todos_sc, y_todos)

print(f'Modelo final treinado com {len(df_rotulado)} registros (2014–2024) ✅')
print(f'Acurácia in-sample: {accuracy_score(y_todos, modelo_final.predict(X_todos_sc)):.4f}')

In [ ]:
# ── Coeficientes do modelo final ────────────────────────────────────
coef_final = pd.DataFrame({
    'Feature':     FEATURES,
    'Coeficiente': modelo_final.coef_[0].round(4),
    'Odds Ratio':  np.exp(modelo_final.coef_[0]).round(4)
}).sort_values('Coeficiente', ascending=False)

print('Coeficientes do Modelo Final (treinado em 2014–2024):')
print(coef_final.to_string(index=False))
print(f'\nIntercepto: {modelo_final.intercept_[0]:.4f}')

# Visualização
fig, ax = plt.subplots(figsize=(9, 3.5))
cores = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_final['Coeficiente']]
bars  = ax.barh(coef_final['Feature'], coef_final['Coeficiente'], color=cores, edgecolor='white', height=0.5)
ax.axvline(x=0, color='black', linewidth=1)
ax.set_title('Coeficientes do Modelo Final — Regressão Logística', fontweight='bold', fontsize=13)
ax.set_xlabel('Coeficiente (log-odds)')
for bar, val in zip(bars, coef_final['Coeficiente']):
    offset = 0.02 if val >= 0 else -0.02
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Salvamento em disco ──────────────────────────────────────────────
MODELO_PATH = os.path.join('modelos', 'modelo_logit_status.pkl')
SCALER_PATH = os.path.join('modelos', 'scaler_logit_status.pkl')

os.makedirs('modelos', exist_ok=True)
joblib.dump(modelo_final, MODELO_PATH)
joblib.dump(scaler_final, SCALER_PATH)

print(f'Modelo salvo → {MODELO_PATH}')
print(f'Scaler salvo → {SCALER_PATH}')

# Verificação: recarrega e testa
m_check = joblib.load(MODELO_PATH)
s_check = joblib.load(SCALER_PATH)
x_test  = s_check.transform(pd.DataFrame({'Plantel': [28], 'Estrangeiros': [4], 'Valor de Mercado Total': [85]}))
print(f'\nVerificação — Previsão para clube médio (Plantel=28, Estrang=4, Valor=85M€):')
print(f'  Classe prevista : {m_check.predict(x_test)[0]} ({"Permaneceu" if m_check.predict(x_test)[0]==1 else "Rebaixado"})')
print(f'  Prob. Rebaixamento: {m_check.predict_proba(x_test)[0][0]:.2%}')
print(f'  Prob. Permanência : {m_check.predict_proba(x_test)[0][1]:.2%}')

---
## 9. Previsão para 2025 <a id='9'></a>

In [ ]:
# ── Dados dos clubes para 2025 ──────────────────────────────────────
df_2025 = df[df['Temporada'] == ANO_PREVISAO].copy()

if df_2025.empty:
    print('⚠️ Nenhum dado encontrado para 2025. Usando dados de 2024 como proxy.')
    df_2025 = df[df['Temporada'] == 2024].copy()

print(f'Clubes com dados para previsão: {len(df_2025)}')
df_2025[['Clube'] + FEATURES].head()

In [ ]:
# ── Aplicando o modelo final para prever 2025 ───────────────────────
X_2025    = df_2025[FEATURES]
X_2025_sc = scaler_final.transform(X_2025)

probs_2025  = modelo_final.predict_proba(X_2025_sc)

# Orienta: classes_[0] deve ser 0 (Rebaixado)
idx_rebaixado = list(modelo_final.classes_).index(0)

df_prev_2025 = df_2025[['Clube'] + FEATURES].copy()
df_prev_2025['Prob_Rebaixamento (%)'] = (probs_2025[:, idx_rebaixado] * 100).round(2)
df_prev_2025 = df_prev_2025.sort_values('Prob_Rebaixamento (%)', ascending=False).reset_index(drop=True)
df_prev_2025.index += 1

# Marcar os 4 primeiros como rebaixados
df_prev_2025['Rebaixado'] = 'Não'
df_prev_2025.loc[df_prev_2025.index <= 4, 'Rebaixado'] = 'Sim'

print('Previsão de Rebaixamento — Brasileirão Série A 2025:')
df_prev_2025[['Clube', 'Prob_Rebaixamento (%)', 'Rebaixado']]

In [ ]:
# ── Gráfico de barras horizontal ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 9))

cores_bar = ['#e74c3c' if r == 'Sim' else '#3498db'
             for r in df_prev_2025['Rebaixado']]

bars = ax.barh(
    df_prev_2025['Clube'][::-1],
    df_prev_2025['Prob_Rebaixamento (%)'][::-1],
    color=cores_bar[::-1], edgecolor='white', height=0.7
)

ax.axvline(x=50, color='gray', linestyle='--', alpha=0.5, label='Limiar 50%')
ax.set_xlabel('Probabilidade de Rebaixamento (%)', fontsize=12)
ax.set_title(f'Previsão de Rebaixamento — Série A {ANO_PREVISAO}',
             fontsize=14, fontweight='bold')

for bar, val in zip(bars, df_prev_2025['Prob_Rebaixamento (%)'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#e74c3c', label='Rebaixado (top 4 risco)'),
    Patch(color='#3498db', label='Permanece na Série A')
], loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# ── Análise de sensibilidade rápida ─────────────────────────────────
# Impacto isolado do Valor de Mercado na probabilidade de rebaixamento
valores = np.linspace(5, 300, 100)
dados_sens = pd.DataFrame({
    'Plantel': 28,
    'Estrangeiros': 4,
    'Valor de Mercado Total': valores
})
probs_sens = modelo_final.predict_proba(scaler_final.transform(dados_sens))[:, idx_rebaixado]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(valores, probs_sens * 100, color='#e74c3c', linewidth=2.5)
ax.fill_between(valores, probs_sens * 100, alpha=0.15, color='#e74c3c')
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.6)
ax.set_xlabel('Valor de Mercado Total (M€)', fontsize=12)
ax.set_ylabel('Prob. de Rebaixamento (%)', fontsize=12)
ax.set_title('Sensibilidade: Valor de Mercado vs Risco de Rebaixamento\n(Plantel=28, Estrangeiros=4)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Análise de Mudança de Situação entre Temporadas <a id='10'></a>

Esta seção identifica **clubes que mudaram de classificação** entre temporadas consecutivas — por exemplo, rebaixados que subiram, ou times que caíram de categoria.

In [ ]:
# ── Padronização da coluna Situacao ─────────────────────────────────
if 'Situacao' in df.columns:
    mapa_situacao = {
        'Top4': 'Top 4',
        'SerieA': 'Série A',
        'SérieA': 'Série A',
        'SerieB_Para_SerieA': 'Série B → Série A',
        'Serie B para Série A': 'Série B → Série A',
        'Rebaixado': 'Rebaixado'
    }
    df['Situacao_label'] = df['Situacao'].replace(mapa_situacao)
else:
    df['Situacao_label'] = df['Status_bin'].map({0: 'Rebaixado', 1: 'Série A'})

print('Distribuição de situações (base completa):')
print(df['Situacao_label'].value_counts())

In [ ]:
# ── Rebaixamentos por temporada ─────────────────────────────────────
df_hist = df[df['Temporada'] < ANO_PREVISAO].copy()
rebaixamentos_por_ano = (
    df_hist[df_hist['Status_bin'] == 0]
    .groupby('Temporada').size()
    .reset_index(name='Rebaixados')
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(rebaixamentos_por_ano['Temporada'], rebaixamentos_por_ano['Rebaixados'],
       color='#e74c3c', edgecolor='white', alpha=0.85)
ax.axhline(y=4, color='gray', linestyle='--', alpha=0.6, label='4 rebaixados/ano (padrão)')
ax.set_title('Número de Rebaixamentos por Temporada', fontsize=13, fontweight='bold')
ax.set_xlabel('Temporada')
ax.set_ylabel('Nº de Clubes Rebaixados')
ax.set_xticks(rebaixamentos_por_ano['Temporada'])
ax.legend()
for bar in ax.patches:
    ax.annotate(f'{int(bar.get_height())}',
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Identificação de mudanças de situação entre temporadas ──────────
df_sorted = df_hist.sort_values(['Clube', 'Temporada']).copy()
df_sorted['Situacao_anterior'] = df_sorted.groupby('Clube')['Situacao_label'].shift(1)
df_sorted['Mudou']             = df_sorted['Situacao_label'] != df_sorted['Situacao_anterior']

df_mudancas = df_sorted[df_sorted['Mudou'] & df_sorted['Situacao_anterior'].notna()].copy()
df_mudancas['Transição'] = df_mudancas['Situacao_anterior'] + ' → ' + df_mudancas['Situacao_label']

print(f'Total de mudanças de situação identificadas: {len(df_mudancas)}')
print()

# Tipos de transição mais frequentes
top_transicoes = df_mudancas['Transição'].value_counts().head(10)
print('Tipos de transição mais frequentes:')
print(top_transicoes.to_string())

In [ ]:
# ── Gráfico de transições ────────────────────────────────────────────
top10 = df_mudancas['Transição'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top10.index[::-1], top10.values[::-1], color='#8e44ad', edgecolor='white', height=0.6)
ax.set_title('Tipos de Transição Mais Frequentes (2014–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequência')
for bar in ax.patches:
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width())}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Tabela detalhada dos casos de rebaixamento ───────────────────────
cols_exib = ['Temporada', 'Clube']
if 'Pontos' in df_hist.columns:
    cols_exib.append('Pontos')
cols_exib += FEATURES + ['Situacao_label']
cols_exib = [c for c in cols_exib if c in df_hist.columns]

df_rebaixados = df_hist[df_hist['Status_bin'] == 0][cols_exib].sort_values('Temporada')
df_rebaixados = df_rebaixados.rename(columns={'Situacao_label': 'Situação'})

print(f'Clubes rebaixados ao longo das temporadas (2014–2024) — {len(df_rebaixados)} casos:')
df_rebaixados

In [ ]:
# ── Clubes com mais rebaixamentos no período ─────────────────────────
mais_rebaixados = (
    df_hist[df_hist['Status_bin'] == 0]
    .groupby('Clube').size()
    .sort_values(ascending=False)
    .reset_index(name='Rebaixamentos')
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(
    mais_rebaixados['Clube'][::-1],
    mais_rebaixados['Rebaixamentos'][::-1],
    color='#c0392b', edgecolor='white', height=0.6
)
ax.set_title('Clubes com Mais Rebaixamentos (2014–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Número de Rebaixamentos')
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
for bar in ax.patches:
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width())}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

---
## Conclusão

Este notebook documentou o pipeline completo de Machine Learning para previsão de rebaixamento no Brasileirão Série A:

1. **Dados**: 240 registros de 2014–2025, coletados no Transfermarkt
2. **Features**: Plantel, Estrangeiros e Valor de Mercado Total
3. **Separação temporal**: Treino 2014–2022 / Teste 2023–2024 (sem vazamento de futuro)
4. **Modelos comparados**: Regressão Logística, Random Forest, SVM
5. **Modelo selecionado**: Regressão Logística (interpretável, alta acurácia)
6. **Modelo final**: Retreinado em 2014–2024, salvo em `modelos/`
7. **Previsão 2025**: Ranking completo dos 20 clubes por risco de rebaixamento

**Para executar o aplicativo web:**
```bash
streamlit run app.py
```

---
*Leonardo Feitosa — Ciência de Dados / UFPB — 2025*